In [1]:
# last update 6/11/2021
# import csv
# import turicreate as tc
import os
import oligo_melting as OligoMelt
import sys
from collections import deque
import RNA
import pandas as pd
import numpy as np
from time import time
scaffold = "GTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGCACCGAGTCGGTGC"
class Features:

    def __init__(self, scaffold, gRNA, dataset=None):
        self.scaffold = scaffold
        self.scaffoldStruct = RNA.fold_compound(scaffold).mfe()[0]
        self.guide = gRNA
        self.mergedStruct = RNA.fold_compound(gRNA + scaffold).mfe()[0]
        self.mergedEnergy = RNA.fold_compound(gRNA + scaffold).mfe()[1]
        self.lstIndexs = []
        self.initHeads()  # find the indexes of the three heads.
        self.firstHeadBegin = self.lstIndexs[0]
        self.firstHeadEnding = self.lstIndexs[1]
        self.secondHeadBegin = self.lstIndexs[2]
        self.secondHeadEnding = self.lstIndexs[3]
        self.thirdHeadBegin = self.lstIndexs[4]
        self.thirdHeadEnding = self.lstIndexs[5]
        self.dataset = dataset

    ''' 
     initHeads - 
     when creating instance of class Features, it will automatically 
     finds the "heads" of the scaffold structure.
    '''

    def initHeads(self):
        index = 0
        myStack = deque()
        while index < len(self.scaffoldStruct) - 1:  # go over scaffold clause struct

            while self.scaffoldStruct[index] != '(':  # locate the beginning index of the next head.
                index += 1
            self.lstIndexs.append(index)
            myStack.append(self.scaffoldStruct[index])
            while myStack:  # as long as the stack is not empty, we didnt locate the ending index of the current head
                index += 1
                if self.scaffoldStruct[index] == '(':
                    myStack.append(self.scaffoldStruct[index])
                if self.scaffoldStruct[index] == ')':
                    myStack.pop()
            self.lstIndexs.append(index)

    '''
    calculate guide energy
    '''

    def guideEnergy(self):
        ene = RNA.fold_compound(self.guide).mfe()[1]
        return ene

    '''
    calculate both guide & scaffold energy
    '''

    def guideAndScaffoldEnergy(self):
        return self.mergedEnergy, self.mergedStruct

    '''
    check if the connection went "well", in other words
    make sure the structure is achieving "base pairs"
    '''

    def basePairs(self):
        guide = self.mergedStruct[0:len(self.guide)]
        myStack = deque()
        index = 0
        while index < len(guide):
            if guide[index] == '(':
                myStack.append(guide[index])
            if guide[index] == ')':
                myStack.pop()
            index += 1

        if myStack:  # if the stack is not empty, means it connected well to the scaffold -> return true
            return True
        else:  # else = stack is empty, connection failed.
            return False

    '''
    confirm first head stays in place
    '''

    def isFirstHeadOk(self):
        originalHead = self.scaffoldStruct[self.lstIndexs[0]:self.lstIndexs[1]]
        newHead = self.mergedStruct[len(self.guide) + self.lstIndexs[0]: len(self.guide) + self.lstIndexs[1]]
        return originalHead == newHead

    '''
    confirm second head stays in place
    '''

    def isSecondHeadOk(self):
        originalHead = self.scaffoldStruct[self.lstIndexs[2]:self.lstIndexs[3]]
        newHead = self.mergedStruct[len(self.guide) + self.lstIndexs[2]: len(self.guide) + self.lstIndexs[3]]
        return originalHead == newHead

    '''
    confirm third head stays in place
    '''

    def isThirdHeadOk(self):
        originalHead = self.scaffoldStruct[self.lstIndexs[4]:self.lstIndexs[5]]
        newHead = self.mergedStruct[len(self.guide) + self.lstIndexs[4]: len(self.guide) + self.lstIndexs[5]]
        return originalHead == newHead

    '''
    confirm all heads stay in place
    '''

    def allHeadAreOk(self):
        return self.isFirstHeadOk() and self.isSecondHeadOk() and self.isThirdHeadOk()

def calculate(seq):
    try:
        seq_as_rna = seq.replace('T', 'U')
        (_, _, _, _, g_RNADNA, _) = OligoMelt.Duplex.calc_tm(seq_as_rna,celsius=True,tt_mode='RNA:DNA')
        (_, _, _, _, g_DNADNA, _) = OligoMelt.Duplex.calc_tm(seq,celsius=True,tt_mode='DNA:DNA')
        g_RNADNA = round(100*g_RNADNA)/100
        g_DNADNA = round(100*g_DNADNA)/100
        # g_DNADNA = Oligo_Melting.calculate_oligo_dna(seq)
        # g_RNADNA = Oligo_Melting.calculate_oligo_rna(seq_as_rna)
        f1 = Features(scaffold, seq)
        # print(f1)
        r1 = round(10*f1.guideEnergy())/10
        r2, r21 = f1.guideAndScaffoldEnergy()
        r2 = round(100*r2)/100
        r3 = f1.basePairs()
        r4 = np.bool(f1.isFirstHeadOk())
        r5 = np.bool(f1.isSecondHeadOk())
        r6 = np.bool(f1.isThirdHeadOk())
        r7 = np.bool(r5 and r6 and r4)
        # can either return results and str or as list.
        result = [g_DNADNA,   g_RNADNA,       r1,         r2,            r21, r3, r4, r5, r6, r7]
                #['g_DNADNA', 'g_RNADNA', 'guideEne', 'guide&scafEne', 'clause',
                   #r3,             r4,       r5,     r6,     r7
                 #'isBasePairs', 'Head1', 'Head2', 'Head3', '1&2&3']
        # result = [seq, r1, r2, r21, r3, r4, r5, r6, r7]
        return result
        # return str(seq) +" " + str(g_DNADNA) + " "+ str(g_RNADNA) + " " + str(r1) + " "+str(r2)+" "+str(r21)+" "+str(r3)+" "+ str(r4) +" "+str(r5)+" "+str(r6)+" "+str(r7)
    except Exception as e:
        print(e)
        return ""

In [2]:
# leenay_all = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Isana_code/Leenay_all.csv')
# leenay_isana = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Isana_code/Leeany_result (1).csv')

In [3]:
# DNAshape_folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/DNAshape_enthalpy_test/current_results/'
# #public data
# DeepCRISPR_hek293 = pd.read_csv(DNAshape_folder+'DeepCRISPR_hek293_w_DNAshape_enthalpy.csv')
# DeepCRISPR_hela = pd.read_csv(DNAshape_folder+'DeepCRISPR_hela_w_DNAshape_enthalpy.csv')
# DeepHF = pd.read_csv(DNAshape_folder+'DeepHF_w_DNAshape_enthalpy.csv')
# Leenay = pd.read_csv(DNAshape_folder+'Leenay_w_DNAshape_enthalpy.csv')
# #Maagad data
#     #human
# ICS =  pd.read_csv(DNAshape_folder+'ICS_w_DNAshape_enthalpy.csv')
# K562 =  pd.read_csv(DNAshape_folder+'K562_w_DNAshape_enthalpy.csv')
# T =  pd.read_csv(DNAshape_folder+'T_w_DNAshape_enthalpy.csv')
# U937 =  pd.read_csv(DNAshape_folder+'U937_w_DNAshape_enthalpy.csv')
#     #tomato
# tomato = pd.read_csv(DNAshape_folder+'tomato_w_DNAshape_enthalpy.csv')
#     #shrimp
# shrimp = pd.read_csv(DNAshape_folder+'Shrimp_w_DNAshape_enthalpy.csv')
# shrimp['target_seq'] = shrimp['gRNA seq']
# all_dfs = [DeepCRISPR_hek293, DeepCRISPR_hela, DeepHF, Leenay, 
#            ICS, K562, T, U937, tomato, shrimp]
# all_dfs_names = ['DeepCRISPR_hek293', 'DeepCRISPR_hela', 'DeepHF', 'Leenay', 
#            'ICS', 'K562', 'T', 'U937', 'tomato', 'Shrimp']
# all_dfs.reverse()
# all_dfs_names.reverse()
# print('done')

In [4]:
isana_cols = ['g_DNADNA','g_RNADNA', 'guideEne', 'guide&scafEne', 'clause', 'isBasePairs', 'Head1', 'Head2', 'Head3', '1&2&3']
isana_cols_types = [np.nan,np.nan,np.nan,np.nan,'',np.bool,np.bool,np.bool,np.bool,np.bool]
def redefine_df(df):
    df = df.drop(columns=[col for col in df.columns if any(x in col for x in isana_cols)])
    new_cols_df = pd.DataFrame({col: col_type  for col,col_type in zip(isana_cols,isana_cols_types)}, index=df.index)
    df = pd.concat([df, new_cols_df], axis=1)
    return df

In [7]:
isana_folder = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Isana_code/'
# path_data = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/DNAshape_enthalpy_test/new_shrimp_fly_tomato/results/'
# list_files = os.listdir(path_data)
# list_files = [x for x in list_files if '.csv' in x]
# df_names = [x.split('_')[0] for x in list_files]
tomato = pd.read_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/Burstein_code/output_sites/fly_new_shrimp_tomato/Tomato_roots_burst.csv')
# list_files = [list_files]
# for filename,organism in zip(list_files,df_names):
#     t=time()
#     print(f'starting {filename}')
#     df = pd.read_csv(os.path.join(path_data, filename))

filename = 'tomato_roots'
for df in [tomato]:
    df = redefine_df(df)
    for i in df.index:
        if i%1000==0:
            print(f'{i/df.shape[0]:.3f}')
        seq = df.loc[i,'target_seq']
        seq = seq[:20]
        seq = seq.upper().replace('U','T')
        res = calculate(seq)
        df.loc[i,isana_cols] = res
    # df = df.drop(columns=[x for x in df.columns if 'Unnamed' in x])
    df.to_csv(isana_folder + 'new_shrimp_fly_tomato/' +f'{filename}_w_isana.csv', index=False)
    print('done')
    # print(time()-t)


0.000
done
